# 📘 Notebook 11: Learning from Pixels

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module C: Deep Value-Based Learning · Notebook 4 of 4 in this module · (11 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what changes when the agent's observation is an **image**
- Build a **convolutional** Q-network and train DQN directly on the pixels of a small game
- Show with an experiment why a **single frame** is not a Markov state, and repair it by **stacking frames**
- Describe the standard **Atari preprocessing**: greyscale, downsampling, frame skipping, frame stacking
- Estimate the **cost** of learning a real video game from the screen, from a measurement on your own machine
- Describe how a well-known project trained an agent to play **Pokémon Red**, and why it needed so much computation

> **Prerequisites:** Notebook 09 (DQN). From *From Python to Deep Learning & Fine-Tuning*, Notebook 14: what a convolutional layer is, and why it suits images. This notebook does not re-teach convolutions.
>
> 🔭 **Why this notebook matters later:** this is the step that made deep reinforcement learning famous: an agent that is given the screen and the score, and nothing else. It closes Module C and it answers the question left open by Snake in Notebook 07, how an agent can find its own features.

> 📦 **Libraries used in this notebook.**
>
> - **PyTorch**, **Gymnasium** and **imageio**, as before.
> - **ale-py**, the Arcade Learning Environment, which contains the Atari 2600 games, and **OpenCV**, which Gymnasium uses to resize the game screen.
>
> No GPU is needed for the cells that run by default. The three training cells take one to three minutes each on an ordinary processor. The last training cell, for Pong, is **optional** and is not run automatically: it needs hours, or a GPU.

In [ ]:
%pip install -q "gymnasium[atari]>=1.3" ale-py opencv-python imageio imageio-ffmpeg torch

In [ ]:
import base64
import copy
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=8, width=240):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. From numbers to pictures

### Intuition first
Every agent so far was handed a tidy description of its world. The snake was told "danger ahead" and "food to the left". The lander received its position and speed as eight numbers. Somebody had to decide what those numbers should be, and for Snake we saw that this decision set the limit of what the agent could learn (Notebook 07).

A person playing a video game gets none of that. They look at the **screen**. The idea of this notebook is to give the agent the same thing: the picture, as an array of numbers, and to let a neural network work out for itself what in the picture matters.

### Formal definition
The observation is an array of **pixels**. The Q-network is a **convolutional neural network (CNN)**: its first layers slide small filters over the picture and detect local patterns, and its last layers turn what was detected into one value per action. Everything else is the DQN of Notebook 09.

### Why it matters
No features are designed by hand, so the same program can learn games that have nothing in common. The price is computation: a picture has many more numbers than a feature vector, the network is larger, and far more experience is needed.

## 2. A small game seen as a picture

To keep training times short we write a miniature video game, small enough to learn in a minute but with the screen as its only observation. A ball falls from the top of a 10 × 10 screen. The agent moves a paddle along the bottom row (left, stay, right) and gets +1 if it catches the ball and −1 if it misses.

The class has two parameters we will need later: `frames`, the number of consecutive pictures in one observation, and `max_speed`, how fast the ball may move sideways.

In [ ]:
class CatchEnv(gym.Env):
    """A ball falls through a small screen. Move the paddle to catch it. The observation is the picture."""

    metadata = {"render_modes": ["rgb_array"], "render_fps": 8}

    def __init__(self, size=10, frames=1, max_speed=0, render_mode="rgb_array"):
        self.size, self.frames, self.max_speed = size, frames, max_speed
        self.render_mode = render_mode
        self.observation_space = gym.spaces.Box(0.0, 1.0, (frames, size, size), dtype=np.float32)
        self.action_space = gym.spaces.Discrete(3)          # 0 left, 1 stay, 2 right

    def _picture(self):
        picture = np.zeros((self.size, self.size), dtype=np.float32)
        picture[self.size - 1, self.paddle - 1:self.paddle + 2] = 0.5      # the paddle is three cells wide
        picture[self.ball_y, self.ball_x] = 1.0
        return picture

    def _observation(self):
        return np.stack(self.history[-self.frames:])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.ball_x = int(self.np_random.integers(self.size))
        self.ball_y = 0
        self.speed = int(self.np_random.integers(-self.max_speed, self.max_speed + 1))   # sideways cells per step
        self.paddle = int(self.np_random.integers(1, self.size - 1))
        self.history = [self._picture()] * self.frames
        return self._observation(), {}

    def step(self, action):
        self.paddle = int(np.clip(self.paddle + action - 1, 1, self.size - 2))
        self.ball_y += 1
        self.ball_x += self.speed
        if self.ball_x < 0 or self.ball_x > self.size - 1:                  # bounce off the side walls
            self.ball_x = -self.ball_x if self.ball_x < 0 else 2 * (self.size - 1) - self.ball_x
            self.speed = -self.speed
        terminated = self.ball_y == self.size - 1
        reward = 0.0
        if terminated:
            reward = 1.0 if abs(self.ball_x - self.paddle) <= 1 else -1.0
        self.history.append(self._picture())
        return self._observation(), reward, terminated, False, {}

    def render(self, cell=24):
        picture = self._picture()
        image = np.zeros((self.size, self.size, 3), dtype=np.uint8) + 20
        image[picture == 1.0] = (240, 200, 40)
        image[picture == 0.5] = (70, 150, 230)
        return np.kron(image, np.ones((cell, cell, 1), dtype=np.uint8))


env = CatchEnv()
observation, info = env.reset(seed=0)
print("shape of an observation:", observation.shape, "(frames, height, width)")
print(observation[0])

That grid of numbers is **all** the agent receives: a 1 where the ball is and 0.5 on the three cells of the paddle. It is not told that there is a ball, or a paddle, or that the two should meet.

`render` draws the same thing enlarged for us to watch.

In [ ]:
def play(env, policy, seed=None, record=False):
    """One game. Returns the final reward (+1 caught, -1 missed) and the frames."""
    observation, info = env.reset(seed=seed)
    frames = [env.render()] if record else []
    terminated = False
    while not terminated:
        observation, reward, terminated, truncated, info = env.step(policy(observation))
        if record:
            frames.append(env.render())
    return reward, frames


def catch_rate(env, policy, games=500):
    return 100 * np.mean([play(env, policy, seed=50000 + game)[0] > 0 for game in range(games)])


random_generator = np.random.default_rng(0)
random_policy = lambda observation: int(random_generator.integers(3))
print(f"random agent: catches the ball in {catch_rate(env, random_policy):.0f}% of games")

frames = []
for game in range(4):
    frames += play(env, random_policy, seed=game, record=True)[1]
show_video(frames)

## 3. A network that reads pictures

A convolutional layer applies the same small filter at every position of the picture. That fits the game: "ball above the left end of the paddle" is the same pattern wherever on the screen it occurs, and a filter that has learned it in one place recognises it everywhere.

The network has two convolutional layers, followed by two ordinary layers that produce the three action values. The number of **input channels** equals the number of frames in an observation.

In [ ]:
def make_cnn(frames, size, n_actions):
    return nn.Sequential(
        nn.Conv2d(frames, 16, kernel_size=3, padding=1), nn.ReLU(),
        nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(),
        nn.Flatten(),
        nn.Linear(32 * size * size, 128), nn.ReLU(),
        nn.Linear(128, n_actions))


torch.manual_seed(0)
network = make_cnn(frames=1, size=10, n_actions=3)
with torch.no_grad():
    values = network(torch.as_tensor(observation)[None])       # [None] adds the batch dimension

print("input shape: ", tuple(torch.as_tensor(observation)[None].shape))
print("output:      ", values.numpy().round(3), "(one value per action)")
print("parameters:  ", sum(p.numel() for p in network.parameters()))

## 4. DQN from pixels

The algorithm is the DQN of Notebook 09 with two substitutions: the replay buffer stores pictures, and the network is the CNN. Compare the function below with the one in Notebook 09, line by line. Nothing in the learning rule knows that the input is an image.

In [ ]:
class ImageBuffer:
    """The replay buffer of Notebook 09, for states that are pictures."""

    def __init__(self, capacity, shape):
        self.states = np.zeros((capacity, *shape), dtype=np.float32)
        self.next_states = np.zeros((capacity, *shape), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.capacity, self.size, self.position = capacity, 0, 0

    def add(self, state, action, reward, next_state, terminated):
        i = self.position
        self.states[i], self.actions[i], self.rewards[i] = state, action, reward
        self.next_states[i], self.terminated[i] = next_state, float(terminated)
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, rng):
        index = rng.integers(0, self.size, batch_size)
        return tuple(torch.as_tensor(array[index]) for array in
                     (self.states, self.actions, self.rewards, self.next_states, self.terminated))


def pixel_dqn(env, steps, learning_rate=5e-4, gamma=0.95, batch_size=32, buffer_size=20000, warm_up=500,
              train_every=2, tau=0.01, explore_steps=10000, epsilon_end=0.02, seed=0, log=None):
    """DQN with a convolutional network. Returns the network and, for every game, whether the ball was caught."""
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    shape, n_actions = env.observation_space.shape, env.action_space.n
    online = make_cnn(shape[0], shape[1], n_actions)
    target = copy.deepcopy(online)
    optimiser = torch.optim.Adam(online.parameters(), lr=learning_rate)
    buffer = ImageBuffer(buffer_size, shape)
    caught = []
    state, info = env.reset(seed=seed)
    for step in range(1, steps + 1):
        epsilon = max(epsilon_end, 1 - step / explore_steps)
        if rng.random() < epsilon:
            action = int(rng.integers(n_actions))
        else:
            with torch.no_grad():
                action = int(online(torch.as_tensor(state)[None]).argmax())
        next_state, reward, terminated, truncated, info = env.step(action)
        buffer.add(state, action, reward, next_state, terminated)
        state = next_state
        if terminated or truncated:
            caught.append(reward > 0)
            state, info = env.reset()
            if log and len(caught) % log == 0:
                print(f"step {step:>6,}   game {len(caught):>5}   caught {100 * np.mean(caught[-log:]):3.0f}% of the last {log}")
        if buffer.size >= warm_up and step % train_every == 0:
            states, actions, rewards, next_states, done = buffer.sample(batch_size, rng)
            with torch.no_grad():
                targets = rewards + gamma * (1 - done) * target(next_states).max(dim=1).values
            predicted = online(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            loss = nn.functional.smooth_l1_loss(predicted, targets)
            optimiser.zero_grad()
            loss.backward()
            optimiser.step()
            with torch.no_grad():
                for w, w_target in zip(online.parameters(), target.parameters()):
                    w_target.mul_(1 - tau).add_(tau * w)
    return online, np.array(caught)


def greedy(network):
    def policy(observation):
        with torch.no_grad():
            return int(network(torch.as_tensor(observation)[None]).argmax())
    return policy

We train for 12,000 steps, about 1,300 games. This takes a minute or so.

In [ ]:
network, caught = pixel_dqn(env, 12000, log=250)

print(f"\ntrained agent: catches the ball in {catch_rate(env, greedy(network)):.0f}% of 500 new games")

frames = []
for game in range(4):
    frames += play(env, greedy(network), seed=game, record=True)[1]
show_video(frames)

From a rate no better than chance the agent has reached an almost perfect one, and the only input was a grid of 100 numbers per step. It was never told which of them is the ball.

## 5. What a single picture cannot show

### Intuition first
Look at a photograph of a tennis ball in mid-air. Where will it be a moment later? You cannot say: the photograph does not show which way the ball is moving, or how fast. Two photographs taken one after the other do.

In the game so far the ball fell straight down, so one picture was enough. Now let it also move **sideways**, by up to two cells per step, bouncing off the walls. The paddle moves only one cell per step, so it cannot simply follow the ball. It has to work out where the ball is **going** and set off in time.

### Formal definition
Notebook 02 defined a Markov state as one that contains everything relevant for the future. A single frame of a moving object is **not** a Markov state: positions are visible, velocities are not. The standard repair is **frame stacking**: the observation consists of the last few frames, as separate channels of the input. Velocity is then present in the observation, as the difference between the channels.

First, an agent that sees one frame at a time:

In [ ]:
one_frame_env = CatchEnv(frames=1, max_speed=2)
one_frame_agent, caught_one = pixel_dqn(one_frame_env, 25000, log=1000)
rate_one = catch_rate(one_frame_env, greedy(one_frame_agent))
print(f"\none frame per observation: catches {rate_one:.0f}%")

Now the same game, the same algorithm and the same amount of training, with **two** frames per observation: the current picture and the one before it.

In [ ]:
two_frame_env = CatchEnv(frames=2, max_speed=2)
two_frame_agent, caught_two = pixel_dqn(two_frame_env, 25000, log=1000)
rate_two = catch_rate(two_frame_env, greedy(two_frame_agent))

print(f"\none frame per observation:  catches {rate_one:.0f}%")
print(f"two frames per observation: catches {rate_two:.0f}%")

frames = []
for game in range(5):
    frames += play(two_frame_env, greedy(two_frame_agent), seed=game, record=True)[1]
show_video(frames)

The agent with one frame stays far from a perfect score however long it trains: for a ball in a given position it cannot know whether to go left or right, and it can do no better than a sensible guess. With two frames the missing information is there, and the same network learns to move to where the ball will land.

> 🧠 This is the same lesson as Snake in Notebook 07, from the other side. There, hand-made features hid part of the board from the agent. Here, a raw picture hides the motion. An agent can only be as good as its observation allows, whichever way the observation is produced.

## 6. The real thing: Atari

The 2015 DQN paper used the games of the Atari 2600 console, through an emulator called the **Arcade Learning Environment**. We look at one of them, **Pong**: two paddles, a ball, first to 21 points wins.

In [ ]:
import ale_py

gym.register_envs(ale_py)

raw_pong = gym.make("ALE/Pong-v5", render_mode="rgb_array", frameskip=1)
frame, info = raw_pong.reset(seed=0)
for step in range(60):
    frame, reward, terminated, truncated, info = raw_pong.step(raw_pong.action_space.sample())

print("one raw frame:", frame.shape, frame.dtype, "=", frame.size, "numbers")
print("actions:", raw_pong.unwrapped.get_action_meanings())

plt.figure(figsize=(3, 4))
plt.imshow(frame)
plt.axis("off")
plt.show()

A raw frame is 210 × 160 pixels in three colours: about a hundred thousand numbers, sixty times per second. Feeding that to a network directly would be wasteful, because most of it is irrelevant. The DQN paper introduced a preprocessing that has been standard ever since:

| Step | What it does | Why |
|---|---|---|
| **Greyscale** | three colour channels become one | colour rarely matters for play |
| **Downsampling** | the picture is shrunk to 84 × 84 | fewer numbers, same information |
| **Frame skipping** | the agent chooses an action every 4th frame, and the action is repeated in between | consecutive frames are nearly identical |
| **Frame stacking** | the observation is the last 4 processed frames | to make motion visible, as in Section 5 |

Gymnasium provides all of it as **wrappers**, objects that are placed around an environment and change what goes in and out.

In [ ]:
pong = gym.make("ALE/Pong-v5", frameskip=1)
pong = gym.wrappers.AtariPreprocessing(pong, frame_skip=4, screen_size=84, grayscale_obs=True)
pong = gym.wrappers.FrameStackObservation(pong, stack_size=4)

observation, info = pong.reset(seed=0)
for step in range(40):
    observation, reward, terminated, truncated, info = pong.step(pong.action_space.sample())

print("one observation:", observation.shape, observation.dtype, "=", observation.size, "numbers")

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(observation[i], cmap="gray")
    ax.set_title(f"frame {i + 1} of 4")
    ax.axis("off")
plt.show()

Look closely at the ball in the four frames: it is in a slightly different place in each. That difference is what tells the network where the ball is heading.

The network of the DQN paper has three convolutional layers and two fully connected ones:

In [ ]:
def make_atari_network(n_actions):
    return nn.Sequential(
        nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
        nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
        nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU(),
        nn.Flatten(),
        nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
        nn.Linear(512, n_actions))


atari_network = make_atari_network(pong.action_space.n)
state = torch.as_tensor(observation, dtype=torch.float32)[None] / 255      # pixel values scaled to 0..1
with torch.no_grad():
    print("action values (untrained):", atari_network(state).numpy().round(3))
print("parameters:", f"{sum(p.numel() for p in atari_network.parameters()):,}")

About 1.7 million parameters, against a few thousand for the lander. And here is how an untrained agent plays:

In [ ]:
video_pong = gym.make("ALE/Pong-v5", render_mode="rgb_array", frameskip=4)
state, info = video_pong.reset(seed=0)
frames, ours, theirs = [], 0, 0
for step in range(600):
    state, reward, terminated, truncated, info = video_pong.step(video_pong.action_space.sample())
    frames.append(video_pong.render())
    ours, theirs = ours + (reward > 0), theirs + (reward < 0)
    if terminated or truncated:
        break

print(f"random agent after {len(frames)} decisions: {ours} points, opponent {theirs} points")
show_video(frames[::2], fps=30, width=320)

## 7. What it costs

Could we train this agent here, as we trained the lander? The honest way to answer is to measure. The cell below times the two things a DQN step consists of, on **your** machine: stepping the game, and one gradient step on a minibatch of 32 observations (taken every 4th step, as in Notebook 09).

In [ ]:
start = time.time()
for step in range(1000):
    observation, reward, terminated, truncated, info = pong.step(pong.action_space.sample())
    if terminated or truncated:
        observation, info = pong.reset()
game_step = (time.time() - start) / 1000

batch = torch.rand(32, 4, 84, 84)
optimiser = torch.optim.Adam(atari_network.parameters(), lr=1e-4)
start = time.time()
for i in range(10):
    loss = atari_network(batch).pow(2).mean()          # a stand-in loss: only the timing matters here
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
gradient_step = (time.time() - start) / 10

per_step = game_step + gradient_step / 4
print(f"one game step:      {1000 * game_step:6.2f} ms")
print(f"one gradient step:  {1000 * gradient_step:6.2f} ms")
print(f"one million agent steps would take about {1e6 * per_step / 3600:.1f} hours on this machine")

DQN typically needs on the order of a million decisions before it starts to win at Pong, and Pong is one of the easiest Atari games. On a processor that means hours. A GPU makes the gradient steps many times faster, which is why this kind of training is done on one.

So this notebook stops short of training Pong by default. If you have the time, or have switched the Colab runtime to a GPU (**Runtime → Change runtime type**), the optional cell below does it with **Stable-Baselines3**, a library of ready-made reinforcement learning algorithms that Notebook 14 introduces properly. We checked that the cell starts and runs. We did not let it run to the end while preparing this notebook, so treat the number of steps as a starting point.

In [ ]:
# OPTIONAL: long. Needs `pip install stable-baselines3` and, realistically, a GPU.
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

sb3_pong = VecFrameStack(make_atari_env("PongNoFrameskip-v4", n_envs=1, seed=0), n_stack=4)
model = DQN("CnnPolicy", sb3_pong, buffer_size=50_000, learning_rate=1e-4, batch_size=32,
            learning_starts=10_000, target_update_interval=1_000, train_freq=4,
            exploration_fraction=0.1, exploration_final_eps=0.01, verbose=1)
model.learn(total_timesteps=1_500_000)
model.save("pong_dqn")

## 8. A famous example: Pokémon Red

In 2023 the programmer Peter Whidden published an open-source project, *Playing Pokemon Red with Reinforcement Learning*, together with a widely watched video about it. An agent learns to play the Game Boy game from the screen, as in this notebook. It is worth knowing how it was done, because it shows both the reach and the cost of the approach.

The published training script tells us the main ingredients:

- **The game runs in an emulator** (PyBoy), which plays the role of our environment classes. The game itself is not part of the project: whoever runs it must supply their own copy.
- **The algorithm is PPO**, from Stable-Baselines3, with a convolutional policy. PPO is the subject of Notebook 14.
- **Sixteen copies of the game run in parallel**, and the experience of all of them trains one network.
- **The agent chooses a button every 24 frames**, a stronger version of the frame skipping of Section 6.
- **An episode lasts about 20,000 decisions**, and the script is configured for a budget of more than a billion decisions in total.

The hardest part was not the algorithm. It was the **reward**. Pokémon gives almost no natural signal: the first badge is thousands of button presses away, and an agent pressing buttons at random will never reach it. The project therefore rewards **exploration**, giving a bonus when the agent reaches a place it has not seen before, in addition to rewards for progress, such as the levels of its Pokémon, that are read from the game's memory. Much of the work went into adjusting those rewards so that the agent would not get stuck exploiting one of them. That is the reward-design problem of Notebook 07, at full scale.

Compare the numbers with ours. The lander needed about 200,000 decisions and a few minutes. Catch needed 12,000. A long game with a huge world and rare rewards needs many orders of magnitude more, and that, more than any missing idea, is why we stop at small games in this course.

> ⚠️ **Common pitfalls**
>
> - **Forgetting to stack frames** when the game has motion. The agent then faces a problem that cannot be solved, and no amount of training helps.
> - **Feeding raw pixel values of 0 to 255 into the network.** Scale them to the range 0 to 1.
> - **A replay buffer of pictures that does not fit in memory.** One preprocessed Atari observation is 28,224 bytes. Exercise 4 does the arithmetic.
> - **Judging feasibility by intuition.** Measure the time per step first, as in Section 7, and multiply.

---
## ✏️ Exercises

The exercises use `CatchEnv`, `make_cnn`, `pixel_dqn`, `greedy`, `catch_rate`, `play` and the trained agents of Sections 4 and 5.

### Exercise 1 (Counting parameters)
`make_cnn(frames=1, size=10, n_actions=3)` has two convolutional layers and two linear layers. Compute the number of parameters of each layer by hand (a convolution with $c_{in}$ input channels, $c_{out}$ filters of size 3 × 3 has $c_{in} \cdot c_{out} \cdot 9 + c_{out}$ parameters) and check your result against PyTorch. Which layer holds most of the parameters? What changes with `frames=2`?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
by_hand = {
    "conv 1":   1 * 16 * 9 + 16,
    "conv 2":   16 * 32 * 9 + 32,
    "linear 1": 32 * 10 * 10 * 128 + 128,
    "linear 2": 128 * 3 + 3,
}
for name, count in by_hand.items():
    print(f"{name:9s} {count:>8,}")
print(f"{'total':9s} {sum(by_hand.values()):>8,}")
print("PyTorch, 1 frame: ", sum(p.numel() for p in make_cnn(1, 10, 3).parameters()))
print("PyTorch, 2 frames:", sum(p.numel() for p in make_cnn(2, 10, 3).parameters()))
```

*Almost all the parameters sit in the first linear layer, which connects every position of every feature map to 128 units. The convolutions are small, because a filter is reused at every position. A second input frame adds only 16 × 9 = 144 parameters, to the first convolution.*
</details>

### Exercise 2 (A slower ball)
In Section 5 the ball moved sideways by up to 2 cells per step and one frame was not enough. Train a one-frame agent on `CatchEnv(frames=1, max_speed=1)` for 25,000 steps and measure its catch rate. Explain why one frame is (nearly) enough here, although the ball still moves sideways.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
slow_env = CatchEnv(frames=1, max_speed=1)
slow_agent, caught_slow = pixel_dqn(slow_env, 25000)
print(f"ball speed up to 1, one frame: catches {catch_rate(slow_env, greedy(slow_agent)):.0f}%")
print(f"ball speed up to 2, one frame: catches {rate_one:.0f}%")
```

*A ball that moves one cell sideways per step is no faster than the paddle. The agent does not need to predict anything: it can keep the paddle under the ball and follow it, and the paddle is three cells wide. Following needs only the current positions, and those are in a single frame. Frame stacking is necessary when the right action depends on the velocity, not whenever something moves.*
</details>

### Exercise 3 (Does the agent really use the motion?)
Take the two-frame agent of Section 5. Play 500 games in which, at every step, you show it the **current frame twice** in place of (previous, current), so that the picture is up to date but contains no motion. Compare its catch rate with the normal one. What does the result prove?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def without_motion(observation):
    frozen = np.stack([observation[1], observation[1]])       # the current frame, twice
    return greedy(two_frame_agent)(frozen)

print(f"two-frame agent, normal input:      catches {catch_rate(two_frame_env, greedy(two_frame_agent)):.0f}%")
print(f"two-frame agent, motion taken away: catches {catch_rate(two_frame_env, without_motion):.0f}%")
```

*Every position is still visible, yet the catch rate drops sharply. The network was not merely looking at where the ball is. It had learned to read the direction of the ball from the difference between its two input channels, without anyone telling it what a velocity is.*
</details>

### Exercise 4 (The memory of a replay buffer)
The original DQN stored one million transitions. Compute the memory needed for one million observations (a) as raw frames of 210 × 160 × 3 bytes, (b) as preprocessed stacks of 4 × 84 × 84 bytes, and (c) as stacks of 4 × 84 × 84 numbers stored as 32-bit floats, which is what our `ImageBuffer` would do. Give the answers in gigabytes. A buffer stores both the state and the next state: what does that do to the numbers, and how could it be avoided?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
million = 1_000_000
options = {
    "raw colour frames (uint8)":        210 * 160 * 3,
    "4 x 84 x 84 stack (uint8)":        4 * 84 * 84,
    "4 x 84 x 84 stack (float32)":      4 * 84 * 84 * 4,
}
for name, size in options.items():
    print(f"{name:30s} {size:>7,} bytes each   {million * size / 1e9:6.1f} GB per million")
```

*Storing state and next state doubles every figure. Even the preprocessed version needs tens of gigabytes that way. Real implementations store each 84 × 84 frame once, as bytes, and rebuild the stacks when sampling: consecutive observations share three of their four frames, and the next state of one transition is the state of the following one. That brings a million transitions down to about 7 GB.*
</details>

### Exercise 5 (A larger screen (a little harder))
Train on a 14 × 14 screen with a ball that falls straight down (`CatchEnv(size=14)`) for 12,000 steps, and train the 10 × 10 game again for comparison, timing both. For each, print the catch rate, the training time, the number of games those 12,000 steps contained, and the number of parameters of the network. What grows with the size of the screen?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for size in (10, 14):
    board = CatchEnv(size=size)
    start = time.time()
    agent, caught_games = pixel_dqn(board, 12000)
    seconds = time.time() - start
    parameters = sum(p.numel() for p in agent.parameters())
    print(f"{size} x {size}: catches {catch_rate(board, greedy(agent)):3.0f}%   {seconds:4.0f} seconds   "
          f"{len(caught_games):>5} games   {parameters:>7,} parameters")
```

*The larger game is still learned, but everything about it costs more. Each observation has twice as many pixels and the network has almost twice as many parameters, so every step takes longer. A game lasts 13 steps and not 9, so the same number of steps contains fewer games, and with them fewer rewards to learn from. Now imagine the step from 14 × 14 to the 84 × 84 of Atari, and from 13 steps per game to thousands.*
</details>

## 🔑 Key takeaways

- An agent can learn from the **screen alone**: a convolutional Q-network replaces the features we used to design by hand.
- The learning rule does not change. DQN from pixels is DQN with a **CNN** and a buffer of pictures.
- A single frame hides **motion**, so it is not a Markov state. **Stacking frames** puts the velocity back into the observation.
- Atari agents use a standard **preprocessing**: greyscale, 84 × 84, an action every 4 frames, a stack of 4 frames.
- Learning from pixels is **expensive**: measure the time per step and multiply before starting.
- Large games such as Pokémon Red add **rare rewards** and long episodes, and need parallel emulators and a very large budget of experience.

### 🏁 You have completed Module C!

You have replaced the table by a network, seen why that breaks Q-learning and how experience replay and a target network repair it, landed a spacecraft, taught a bird to fly through pipes, and trained an agent on raw pixels.

All of these agents learn **values** and act greedily on them. That works when the actions are few. **Module D** takes the other road: it learns the **policy** itself, which opens the way to continuous actions and to the algorithms behind most of today's large reinforcement learning systems.

---
*End of Module C.*

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*